# Importing necessary Libraries

In [ ]:
# warnings
import warnings
warnings.filterwarnings('ignore')

# Libraries for data manipulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set()

# display options
pd.set_option('display.float_format',lambda x:'%.2f'%x)
pd.set_option('display.max_columns',None)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore
import scipy.stats as stats

# models
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.decomposition import PCA

#model_metrics
import sklearn.metrics as metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    precision_recall_curve,
    confusion_matrix,
    make_scorer
    )

# Loading the dataset

In [ ]:
df_travel_train=pd.read_csv('Traveldata_train.csv')
df_survey_train=pd.read_csv('Surveydata_train.csv')
df_travel_test=pd.read_csv('Traveldata_test.csv')
df_survey_test=pd.read_csv('Surveydata_test.csv')

In [ ]:
df_survey_train.isnull().sum()

,0
ID,0
Overall_Experience,0
Seat_Comfort,61
Seat_Class,0
Arrival_Time_Convenient,8930
Catering,8741
Platform_Location,30
Onboard_Wifi_Service,30
Onboard_Entertainment,18
Online_Support,91


In [ ]:
df_travel_train.isnull().sum()

,0
ID,0
Gender,77
Customer_Type,8951
Age,33
Type_Travel,9226
Travel_Class,0
Travel_Distance,0
Departure_Delay_in_Mins,57
Arrival_Delay_in_Mins,357


In [ ]:
train_data=pd.merge(df_travel_train,df_survey_train,on='ID')
test_data=pd.merge(df_travel_test,df_survey_test,on='ID',how='left')

In [ ]:
train_data.isnull().sum()

,0
ID,0
Gender,77
Customer_Type,8951
Age,33
Type_Travel,9226
Travel_Class,0
Travel_Distance,0
Departure_Delay_in_Mins,57
Arrival_Delay_in_Mins,357
Overall_Experience,0


In [ ]:
df_train=train_data.copy()
df_test=test_data.copy()

In [ ]:
# train data
missing_values_train=df_train.isnull().sum()[df_train.isnull().sum()>0]/df_train.shape[0]*100
missing_values_train=missing_values_train.reset_index(drop=False)
missing_values_train.columns=['Columns','Missing%']
missing_values_train=missing_values_train.sort_values(by='Missing%',ascending=False)

In [ ]:
# Test data
missing_values_test=df_test.isnull().sum()[df_test.isnull().sum()>0]/df_test.shape[0]*100
missing_values_test=missing_values_test.reset_index(drop=False)
missing_values_test.columns=['Columns','Missing%']
missing_values_test=missing_values_test.sort_values(by='Missing%',ascending=False)

In [ ]:
# dropping ID
df_train.drop('ID',axis=1,inplace=True)
df_test.drop('ID',axis=1,inplace=True)

# Data Preprocessing

## Feature Engineering

#### Train data

In [ ]:
df_train_fearure_enginering=df_train.copy()

In [ ]:
df_train_fearure_enginering['Departure_Delay_in_hours']=df_train_fearure_enginering['Departure_Delay_in_Mins']/60

In [ ]:
df_train_fearure_enginering['Arrival_Delay_in_hours']=df_train_fearure_enginering['Arrival_Delay_in_Mins']/60

In [ ]:
df_train_fearure_enginering.drop(columns=['Seat_Class'],axis=1,inplace=True)

In [ ]:
df_train_fearure_enginering.drop(columns=['Departure_Delay_in_Mins','Arrival_Delay_in_Mins'],axis=1,inplace=True)

In [ ]:
df_train_fearure_enginering.shape

(73910, 23)

In [ ]:
df_test_fearure_enginering=df_test.copy()

In [ ]:
df_test_fearure_enginering['Departure_Delay_in_hours']=df_test_fearure_enginering['Departure_Delay_in_Mins']/60

In [ ]:
df_test_fearure_enginering['Arrival_Delay_in_hours']=df_test_fearure_enginering['Arrival_Delay_in_Mins']/60

In [ ]:
df_test_fearure_enginering.drop(columns=['Seat_Class'],axis=1,inplace=True)

In [ ]:
df_test_fearure_enginering.drop(columns=['Departure_Delay_in_Mins','Arrival_Delay_in_Mins'],axis=1,inplace=True)

In [ ]:
df_test_fearure_enginering.shape

(35602, 22)

## Missing value treatment

In [ ]:
cat_cols=df_train_fearure_enginering.select_dtypes(exclude=np.number).columns

In [ ]:
df_train_imputed=df_train_fearure_enginering.copy()

In [ ]:
df_test_imputed=df_test_fearure_enginering.copy()

In [ ]:
for i in cat_cols:
    df_train_imputed[i]=df_train_imputed[i].fillna(df_train_imputed[i].mode()[0])
    df_test_imputed[i]=df_test_imputed[i].fillna(df_test_imputed[i].mode()[0])

In [ ]:
#Since Age is almost normally distributed we are imputing with mean
df_train_imputed['Age']=df_train_imputed['Age'].fillna(df_train_imputed['Age'].mean())
df_test_imputed['Age']=df_test_imputed['Age'].fillna(df_test_imputed['Age'].mean())

In [ ]:
#Since below columns are highly right skewed we are imputing with median
df_train_imputed['Departure_Delay_in_hours']=df_train_imputed['Departure_Delay_in_hours'].fillna(df_train_imputed['Departure_Delay_in_hours'].median())
df_train_imputed['Arrival_Delay_in_hours']=df_train_imputed['Arrival_Delay_in_hours'].fillna(df_train_imputed['Arrival_Delay_in_hours'].median())
df_test_imputed['Departure_Delay_in_hours']=df_test_imputed['Departure_Delay_in_hours'].fillna(df_test_imputed['Departure_Delay_in_hours'].median())
df_test_imputed['Arrival_Delay_in_hours']=df_test_imputed['Arrival_Delay_in_hours'].fillna(df_test_imputed['Arrival_Delay_in_hours'].median())

### Data Preparation for Modeling

In [ ]:
y=train_data['Overall_Experience']
y=y.astype(int)

In [ ]:
X=df_train_imputed.copy()
y=X['Overall_Experience']
X=X.drop('Overall_Experience',axis=1)

In [ ]:
X=pd.get_dummies(X,drop_first=True)
df_test_dummies=pd.get_dummies(df_test_imputed,drop_first=True)

In [ ]:
X=X.astype(float)

In [ ]:
X.head()

,Age,Travel_Distance,Departure_Delay_in_hours,Arrival_Delay_in_hours,Gender_Male,Customer_Type_Loyal Customer,Type_Travel_Personal Travel,Travel_Class_Eco,Seat_Comfort_Excellent,Seat_Comfort_Extremely Poor,Seat_Comfort_Good,Seat_Comfort_Needs Improvement,Seat_Comfort_Poor,Arrival_Time_Convenient_Excellent,Arrival_Time_Convenient_Extremely Poor,Arrival_Time_Convenient_Good,Arrival_Time_Convenient_Needs Improvement,Arrival_Time_Convenient_Poor,Catering_Excellent,Catering_Extremely Poor,Catering_Good,Catering_Needs Improvement,Catering_Poor,Platform_Location_Inconvenient,Platform_Location_Manageable,Platform_Location_Needs Improvement,Platform_Location_Very Convenient,Platform_Location_Very Inconvenient,Onboard_Wifi_Service_Excellent,Onboard_Wifi_Service_Extremely Poor,Onboard_Wifi_Service_Good,Onboard_Wifi_Service_Need,Onboard_Wifi_Service_Needs Improvement,Onboard_Wifi_Service_Poor,Onboard_Entertainment_Excellent,Onboard_Entertainment_Extremely Poor,Onboard_Entertainment_Good,Onboard_Entertainment_Needs Improvement,Onboard_Entertainment_Poor,Online_Support_Excellent,Online_Support_Extremely Poor,Online_Support_Good,Online_Support_Needs Improvement,Online_Support_Poor,Ease_of_Online_Booking_Excellent,Ease_of_Online_Booking_Extremely Poor,Ease_of_Online_Booking_Good,Ease_of_Online_Booking_Needs Improvement,Ease_of_Online_Booking_Poor,Onboard_Service_Excellent,Onboard_Service_Extremely Poor,Onboard_Service_Good,Onboard_Service_Needs Improvement,Onboard_Service_Poor,Legroom_Excellent,Legroom_Extremely Poor,Legroom_Good,Legroom_Needs Improvement,Legroom_Poor,Baggage_Handling_Excellent,Baggage_Handling_Good,Baggage_Handling_Needs Improvement,Baggage_Handling_Poor,CheckIn_Service_Excellent,CheckIn_Service_Extremely Poor,CheckIn_Service_Good,CheckIn_Service_Needs Improvement,CheckIn_Service_Poor,Cleanliness_Excellent,Cleanliness_Extremely Poor,Cleanliness_Good,Cleanliness_Needs Improvement,Cleanliness_Poor,Online_Boarding_Excellent,Online_Boarding_Extremely Poor,Online_Boarding_Good,Online_Boarding_Needs Improvement,Online_Boarding_Poor
0,52.00,272.00,0.00,0.08,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00
1,48.00,2200.00,0.15,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
2,43.00,1061.00,1.28,1.98,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00
3,44.00,780.00,0.22,0.30,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,50.00,1981.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.0

#### Splitting data into training, validation

In [ ]:
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.10,stratify=y,random_state=1)

In [ ]:
print("Number of rows in train data =", X_train.shape[0])
print("Number of rows in validation data =", X_val.shape[0])
print("Number of rows in validation data =", df_test_fearure_enginering.shape[0])

Number of rows in train data = 66519
Number of rows in validation data = 7391
Number of rows in validation data = 35602


In [ ]:
print("Number of rows in train data =", y_train.value_counts(1))
print("Number of rows in validation data =", y_val.value_counts(1))

Number of rows in train data = Overall_Experience
1   0.55
0   0.45
Name: proportion, dtype: float64
Number of rows in validation data = Overall_Experience
1   0.55
0   0.45
Name: proportion, dtype: float64


### Scaling

In [ ]:
X_train_scaled=X_train.copy()
X_val_scaled=X_val.copy()
df_test_scaled=df_test_dummies.copy()

In [ ]:
scale_cols=['Age','Travel_Distance','Departure_Delay_in_hours','Arrival_Delay_in_hours']

In [ ]:
scaler=StandardScaler()

In [ ]:
X_train_scaled[scale_cols]=scaler.fit_transform(X_train_scaled[scale_cols])
X_val_scaled[scale_cols]=scaler.fit_transform(X_val_scaled[scale_cols])
df_test_scaled[scale_cols]=scaler.fit_transform(df_test_scaled[scale_cols])

In [ ]:
X_train_scaled.head()

,Age,Travel_Distance,Departure_Delay_in_hours,Arrival_Delay_in_hours,Gender_Male,Customer_Type_Loyal Customer,Type_Travel_Personal Travel,Travel_Class_Eco,Seat_Comfort_Excellent,Seat_Comfort_Extremely Poor,Seat_Comfort_Good,Seat_Comfort_Needs Improvement,Seat_Comfort_Poor,Arrival_Time_Convenient_Excellent,Arrival_Time_Convenient_Extremely Poor,Arrival_Time_Convenient_Good,Arrival_Time_Convenient_Needs Improvement,Arrival_Time_Convenient_Poor,Catering_Excellent,Catering_Extremely Poor,Catering_Good,Catering_Needs Improvement,Catering_Poor,Platform_Location_Inconvenient,Platform_Location_Manageable,Platform_Location_Needs Improvement,Platform_Location_Very Convenient,Platform_Location_Very Inconvenient,Onboard_Wifi_Service_Excellent,Onboard_Wifi_Service_Extremely Poor,Onboard_Wifi_Service_Good,Onboard_Wifi_Service_Need,Onboard_Wifi_Service_Needs Improvement,Onboard_Wifi_Service_Poor,Onboard_Entertainment_Excellent,Onboard_Entertainment_Extremely Poor,Onboard_Entertainment_Good,Onboard_Entertainment_Needs Improvement,Onboard_Entertainment_Poor,Online_Support_Excellent,Online_Support_Extremely Poor,Online_Support_Good,Online_Support_Needs Improvement,Online_Support_Poor,Ease_of_Online_Booking_Excellent,Ease_of_Online_Booking_Extremely Poor,Ease_of_Online_Booking_Good,Ease_of_Online_Booking_Needs Improvement,Ease_of_Online_Booking_Poor,Onboard_Service_Excellent,Onboard_Service_Extremely Poor,Onboard_Service_Good,Onboard_Service_Needs Improvement,Onboard_Service_Poor,Legroom_Excellent,Legroom_Extremely Poor,Legroom_Good,Legroom_Needs Improvement,Legroom_Poor,Baggage_Handling_Excellent,Baggage_Handling_Good,Baggage_Handling_Needs Improvement,Baggage_Handling_Poor,CheckIn_Service_Excellent,CheckIn_Service_Extremely Poor,CheckIn_Service_Good,CheckIn_Service_Needs Improvement,CheckIn_Service_Poor,Cleanliness_Excellent,Cleanliness_Extremely Poor,Cleanliness_Good,Cleanliness_Needs Improvement,Cleanliness_Poor,Online_Boarding_Excellent,Online_Boarding_Extremely Poor,Online_Boarding_Good,Online_Boarding_Needs Improvement,Online_Boarding_Poor
42864,-0.62,2.59,-0.28,-0.39,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
1508,-0.75,0.64,-0.38,-0.28,1.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
71612,-1.02,0.01,-0.38,-0.39,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00
33337,-0.55,0.07,-0.38,-0.39,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
10392,0.44,0.85,-0.38,-0.39,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.0

# Model Building

### data for model building

In [ ]:
X_train_model=X_train_scaled.copy()
X_val_model=X_val_scaled.copy()
df_test_model=df_test_scaled.copy()

In [ ]:
missing_columns=[]

for i in X_train_model:
    if i in df_test_model:
        continue
    else:
        missing_columns.append(i)

In [ ]:
for i in missing_columns:
    df_test_model[i]=0

In [ ]:
X_train_model=X_train_model[sorted(X_train_model.columns)]

In [ ]:
X_val_model=X_val_model[sorted(X_val_model.columns)]

In [ ]:
df_test_model=df_test_model[sorted(df_test_model.columns)]

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn
def model_performance_classification_sklearn(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def confusion_matrix_sklearn(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show()

## Under sample data

In [ ]:
un=RandomUnderSampler(random_state=1,sampling_strategy=1)

In [ ]:
X_train_us,y_train_us=un.fit_resample(X_train_model,y_train)

In [ ]:
print( y_train.value_counts())
print( y_train_us.value_counts())

Overall_Experience
1    46434
0    38507
Name: count, dtype: int64
Overall_Experience
0    38507
1    38507
Name: count, dtype: int64


## Over sample data

In [ ]:
sm=SMOTE(sampling_strategy=1,random_state=1,k_neighbors=5)
X_train_over,y_train_over=sm.fit_resample(X_train_model,y_train)

In [ ]:
print(y_train.value_counts())
print(y_train_over.value_counts())

Overall_Experience
1    46434
0    38507
Name: count, dtype: int64
Overall_Experience
1    46434
0    46434
Name: count, dtype: int64


## Base Model with actual data

In [ ]:
base_models = []  # Empty list to store all the models

# Appending models into the list
base_models.append(("Logistic", LogisticRegression(random_state=1)))
base_models.append(("GaussianNB", GaussianNB()))
base_models.append(("KNeighborsClassifier", KNeighborsClassifier(n_neighbors=5)))
base_models.append(("LDA", LinearDiscriminantAnalysis()))
base_models.append(("dtree", DecisionTreeClassifier(random_state=1)))
base_models.append(("Bagging", BaggingClassifier(random_state=1)))
base_models.append(("Random forest", RandomForestClassifier(random_state=1)))
base_models.append(("GBM", GradientBoostingClassifier(random_state=1)))
base_models.append(("Adaboost", AdaBoostClassifier(random_state=1)))
base_models.append(("xgboost", XGBClassifier(random_state=1)))

print("\nTraining Performance:\n")
for name, model in base_models:
    model.fit(X_train_model, y_train)
    scores = accuracy_score(y_train, model.predict(X_train_model))
    print("{}: {}".format(name, scores))

print("\nValidation Performance:\n")
for name, model in base_models:
    model.fit(X_train_model, y_train)
    scores_val = accuracy_score(y_val, model.predict(X_val_model))
    print("{}: {}".format(name, scores_val))


Training Performance:

Logistic: 0.9023136246786633
GaussianNB: 0.7751168838978337
KNeighborsClassifier: 0.9458199912806867
LDA: 0.8951878410679656
dtree: 1.0
Bagging: 0.9962567086095702
Random forest: 1.0
GBM: 0.9187901201160571
Adaboost: 0.8762609179332221
xgboost: 0.9715870653497497

Validation Performance:

Logistic: 0.9008253281017453
GaussianNB: 0.7754025165742119
KNeighborsClassifier: 0.9186848870247598
LDA: 0.8936544445947774
dtree: 0.9285617643079421
Bagging: 0.943850629143553
Random forest: 0.9492626166959816
GBM: 0.9180083885807062
Adaboost: 0.8728182925179272
xgboost: 0.9527804086050602


In [ ]:
# XGBOOST has the best score so making prediction with test data.

In [ ]:
xgb=XGBClassifier(random_state=1)
xgb.fit(X_train_model, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=1, ...)

In [ ]:
xgb_pred=xgb.predict(df_test_model)

In [ ]:
a=pd.DataFrame(xgb_pred,columns=['Overall_Experience'])
a['ID']=test_data['ID']
base_xgb=a[['ID','Overall_Experience']]
#base_xgb.to_csv('Submission_4.csv',index=False)

## Base Model with Under sample data

In [ ]:
base_models_un = []  # Empty list to store all the models

# Appending models into the list
base_models_un.append(("Logistic", LogisticRegression(random_state=1)))
base_models_un.append(("GaussianNB", GaussianNB()))
base_models_un.append(("KNeighborsClassifier", KNeighborsClassifier(n_neighbors=5)))
base_models_un.append(("LDA", LinearDiscriminantAnalysis()))
base_models_un.append(("dtree", DecisionTreeClassifier(random_state=1)))
base_models_un.append(("Bagging", BaggingClassifier(random_state=1)))
base_models_un.append(("Random forest", RandomForestClassifier(random_state=1)))
base_models_un.append(("GBM", GradientBoostingClassifier(random_state=1)))
base_models_un.append(("Adaboost", AdaBoostClassifier(random_state=1)))
base_models_un.append(("xgboost", XGBClassifier(random_state=1)))

print("\nTraining Performance:\n")
for name, model in base_models_un:
    model.fit(X_train_us,y_train_us)
    scores = accuracy_score(y_train_us, model.predict(X_train_us))
    print("{}: {}".format(name, scores))

print("\nValidation Performance:\n")
for name, model in base_models_un:
    model.fit(X_train_us,y_train_us)
    scores_val = accuracy_score(y_val, model.predict(X_val_model))
    print("{}: {}".format(name, scores_val))

In [ ]:
xgb_under=XGBClassifier(random_state=1)
xgb_under.fit(X_train_us,y_train_us)

In [ ]:
xgb_under_pred=xgb_under.predict(df_test_model)

In [ ]:
a=pd.DataFrame(xgb_under_pred,columns=['Overall_Experience'])
a['ID']=test_data['ID']
base_xgb_under=a[['ID','Overall_Experience']]
#base_xgb_under.to_csv('Submission_5.csv',index=False)

## Base Model with Over sample data

In [ ]:
base_models_over = []  # Empty list to store all the models

# Appending models into the list
base_models_over.append(("Logistic", LogisticRegression(random_state=1)))
base_models_over.append(("GaussianNB", GaussianNB()))
base_models_over.append(("KNeighborsClassifier", KNeighborsClassifier(n_neighbors=5)))
base_models_over.append(("LDA", LinearDiscriminantAnalysis()))
base_models_over.append(("dtree", DecisionTreeClassifier(random_state=1)))
base_models_over.append(("Bagging", BaggingClassifier(random_state=1)))
base_models_over.append(("Random forest", RandomForestClassifier(random_state=1)))
base_models_over.append(("GBM", GradientBoostingClassifier(random_state=1)))
base_models_over.append(("Adaboost", AdaBoostClassifier(random_state=1)))
base_models_over.append(("xgboost", XGBClassifier(random_state=1)))

print("\nTraining Performance:\n")
for name, model in base_models_over:
    model.fit(X_train_over,y_train_over)
    scores = accuracy_score(y_train_over, model.predict(X_train_over))
    print("{}: {}".format(name, scores))

print("\nValidation Performance:\n")
for name, model in base_models_over:
    model.fit(X_train_over,y_train_over)
    scores_val = accuracy_score(y_val, model.predict(X_val_model))
    print("{}: {}".format(name, scores_val))

# Hypertuning

### Tuning XGBoostClassifier model with Actual data

In [ ]:
%%time

xgb_tuned = XGBClassifier(random_state=1)

parameters={
            'learning_rate': [0.01, 0.1, 0.2],
           }
# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(xgb_tuned, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_model,y_train)

xgb_tuned=randomized_cv.best_estimator_

xgb_tuned.fit(X_train_model,y_train)

In [ ]:
confusion_matrix_sklearn(xgb_tuned,X_train_model,y_train)

In [ ]:
xgb_tuned_train_perf = model_performance_classification_sklearn(
    xgb_tuned,X_train_model,y_train
)
xgb_tuned_train_perf

In [ ]:
confusion_matrix_sklearn(xgb_tuned, X_val_model, y_val)

In [ ]:
xgb_tuned_val_perf = model_performance_classification_sklearn(
    xgb_tuned,X_val_model,y_val
)
xgb_tuned_val_perf

In [ ]:
accuracy_score(y_val,xgb_tuned.predict(X_val_model))

In [ ]:
xgb_pred=xgb_tuned.predict(df_test_model)
a=pd.DataFrame(xgb_pred,columns=['Overall_Experience'])
a['ID']=test_data['ID']
xgb_tuned=a[['ID','Overall_Experience']]
xgb_tuned.to_csv('Submission_9.csv',index=False)

### Tuning XGBoostClassifier model with under sample data

In [ ]:
%%time

xgb_tuned_us = XGBClassifier(random_state=1)

parameters={
            'n_estimators':[100,150,200],
           }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(xgb_tuned_us, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_us, y_train_us)

xgb_tuned_us=randomized_cv.best_estimator_

xgb_tuned_us.fit(X_train_us, y_train_us)

In [ ]:
confusion_matrix_sklearn(xgb_tuned_us,X_train_us, y_train_us)

In [ ]:
xgb_tuned_us_train_perf = model_performance_classification_sklearn(
    xgb_tuned_us,X_train_us, y_train_us
)
xgb_tuned_us_train_perf

In [ ]:
confusion_matrix_sklearn(xgb_tuned_us, X_val_model, y_val)

In [ ]:
xgb_tuned_us_val_perf = model_performance_classification_sklearn(
    xgb_tuned_us,X_val_model,y_val
)
xgb_tuned_us_val_perf

In [ ]:
accuracy_score(y_val,xgb_tuned_us.predict(X_val_model))

In [ ]:
xgb_pred_under=xgb_tuned_us.predict(df_test_model)
a=pd.DataFrame(xgb_pred_under,columns=['Overall_Experience'])
a['ID']=test_data['ID']
xgb_tuned_under=a[['ID','Overall_Experience']]
xgb_tuned_under.to_csv('Submission_8.csv',index=False)

### Tuning XGBoostClassifier model with Over sample data

In [ ]:
%%time

xgb_tuned_over = XGBClassifier(random_state=1)

'''parameters={
            'n_estimators':np.arange(50,110,5),
            'subsample':[0.6,0.7,0.8,0.9,1],
#            'learning_rate':[0.001,0.01,0.1,1],
#            'gamma':[5,6,7,8,9,10],
             'max_depth':[5,6,7,8,9],
             'colsample_bytree':[0.6,0.7,0.8,0.9]
           }
'''
parameters={
            'n_estimators':[50,100,200]
           }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(xgb_tuned_over, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_over, y_train_over)

xgb_tuned_over=randomized_cv.best_estimator_

xgb_tuned_over.fit(X_train_over, y_train_over)

In [ ]:
confusion_matrix_sklearn(xgb_tuned_over,X_train_over,y_train_over)

In [ ]:
xgb_tuned_over_train_perf = model_performance_classification_sklearn(
    xgb_tuned_over,X_train_over, y_train_over
)
xgb_tuned_over_train_perf

In [ ]:
confusion_matrix_sklearn(xgb_tuned_over,X_val_model,y_val)

In [ ]:
xgb_tuned_over_val_perf = model_performance_classification_sklearn(
    xgb_tuned_over,X_val_model,y_val
)
xgb_tuned_over_val_perf

In [ ]:
accuracy_score(y_val,xgb_tuned_over.predict(X_val_model))

In [ ]:
a=pd.DataFrame(xgb_tuned_over_pred,columns=['Overall_Experience'])
a['ID']=test_data['ID']
xgb_tuned=a[['ID','Overall_Experience']]
#xgb_tuned.to_csv('Submission_6.csv',index=False)

# Tuning RandomForest model with normal data

In [ ]:
%%time

rf_tuned = RandomForestClassifier(random_state=1,class_weight='balanced')

parameters={
            'n_estimators':np.arange(10,110,10),
            'max_samples':[0.7,0.8,0.9],
            'max_features':[0.7,0.8,0.9],
            'n_jobs':[-1],
           }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(rf_tuned, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_model, y_train)

rf_tuned=randomized_cv.best_estimator_

rf_tuned.fit(X_train_model, y_train)

In [ ]:
confusion_matrix_sklearn(rf_tuned,X_train_model,y_train)

In [ ]:
rf_tuned_train_perf = model_performance_classification_sklearn(
    rf_tuned,X_train_model,y_train
)
rf_tuned_train_perf

In [ ]:
confusion_matrix_sklearn(rf_tuned, X_val_model, y_val)

In [ ]:
rf_tuned_val_perf = model_performance_classification_sklearn(
    rf_tuned,X_val_model,y_val
)
rf_tuned_val_perf

In [ ]:
accuracy_score(y_val,rf_tuned.predict(X_val_model))

# Tuning RandomForest model with under sampled data

In [ ]:
%%time

rf_tuned_under = RandomForestClassifier(random_state=1,class_weight='balanced',n_jobs=-1)

parameters={
            'n_estimators':np.arange(10,110,10),
            'max_samples':[0.7,0.8,0.9],
            'max_features':[0.7,0.8,0.9]
           }

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(rf_tuned_under, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_us, y_train_us)

rf_tuned_under=randomized_cv.best_estimator_

rf_tuned_under.fit(X_train_us, y_train_us)

In [ ]:
confusion_matrix_sklearn(rf_tuned_under,X_train_us, y_train_us)

In [ ]:
rf_tuned_train_us_perf = model_performance_classification_sklearn(
    rf_tuned_under,X_train_us, y_train_us
)
rf_tuned_train_us_perf

In [ ]:
confusion_matrix_sklearn(rf_tuned_under, X_val_model, y_val)

In [ ]:
rf_tuned_val_us_perf = model_performance_classification_sklearn(
    rf_tuned_under,X_val_model,y_val
)
rf_tuned_val_us_perf

In [ ]:
accuracy_score(y_val,rf_tuned_under.predict(X_val_model))

# Tuning DecisionTree model with normal data

In [ ]:
%%time

dt_tuned = DecisionTreeClassifier(random_state=1)

parameters = {
#    "class_weight": [None, "balanced"],
#    "max_depth": np.arange(2, 11, 2),
    "max_leaf_nodes": [50, 75, 150, 250],
#    "min_samples_split": [10, 30, 50, 70],
    "max_features":[0.7,0.8,0.9,1]
}
# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(dt_tuned, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_model, y_train)

dt_tuned=randomized_cv.best_estimator_

dt_tuned.fit(X_train_model, y_train)

In [ ]:
confusion_matrix_sklearn(dt_tuned,X_train_model,y_train)

In [ ]:
dt_tuned_train_perf = model_performance_classification_sklearn(
    dt_tuned,X_train_model,y_train
)
dt_tuned_train_perf

In [ ]:
confusion_matrix_sklearn(dt_tuned,X_val_model,y_val)

In [ ]:
dt_tuned_val_perf = model_performance_classification_sklearn(
    dt_tuned,X_val_model,y_val
)
dt_tuned_val_perf

# Tuning DecisionTree model with under sample data

In [ ]:
%%time

dt_tuned_under = DecisionTreeClassifier(random_state=1)

parameters = {
#    "class_weight": [None, "balanced"],
#    "max_depth": np.arange(2, 11, 2),
    "max_leaf_nodes": [50, 75, 150, 250],
#    "min_samples_split": [10, 30, 50, 70],
    "max_features":[0.7,0.8,0.9,1]
}
# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(dt_tuned_under, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_us, y_train_us)

dt_tuned_under=randomized_cv.best_estimator_

dt_tuned_under.fit(X_train_us, y_train_us)

In [ ]:
confusion_matrix_sklearn(dt_tuned_under,X_train_us,y_train_us)

In [ ]:
dt_tuned_train_under_perf = model_performance_classification_sklearn(
    dt_tuned_under,X_train_us,y_train_us
)
dt_tuned_train_under_perf

In [ ]:
confusion_matrix_sklearn(dt_tuned_under,X_val_model,y_val)

In [ ]:
dt_tuned_val_under_perf = model_performance_classification_sklearn(
    dt_tuned_under,X_val_model,y_val
)
dt_tuned_val_under_perf

# Bagging

In [ ]:
%%time

bc_tuned = BaggingClassifier(random_state=1)

parameters = {
    "estimator":[LogisticRegression(random_state=1),DecisionTreeClassifier(random_state=1)],
    "max_features":[0.7,0.8,0.9,1],
    "max_samples":[0.7,0.8,0.9,1]
}
# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(bc_tuned, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_model, y_train)

bc_tuned=randomized_cv.best_estimator_

bc_tuned.fit(X_train_model, y_train)

In [ ]:
confusion_matrix_sklearn(bc_tuned,X_train_model,y_train)

In [ ]:
bc_tuned_train_perf = model_performance_classification_sklearn(
    bc_tuned,X_train_model,y_train
)
bc_tuned_train_perf

In [ ]:
confusion_matrix_sklearn(bc_tuned,X_val_model,y_val)

In [ ]:
bc_tuned_val_perf = model_performance_classification_sklearn(
    bc_tuned,X_val_model,y_val
)
bc_tuned_val_perf

In [ ]:
accuracy_score(y_val,bc_tuned.predict(X_val_model))

In [ ]:
%%time

bc_tuned_under = BaggingClassifier(random_state=1)

parameters = {
    "estimator":[LogisticRegression(random_state=1),DecisionTreeClassifier(random_state=1)],
    "max_features":[0.7,0.8,0.9,1],
    "max_samples":[0.7,0.8,0.9,1]
}
# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(bc_tuned_under, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_us, y_train_us)

bc_tuned_under=randomized_cv.best_estimator_

bc_tuned_under.fit(X_train_us, y_train_us)

In [ ]:
confusion_matrix_sklearn(bc_tuned_under,X_train_us,y_train_us)

In [ ]:
bc_tuned_train_under_perf = model_performance_classification_sklearn(
    bc_tuned_under,X_train_us,y_train_us
)
bc_tuned_train_under_perf

In [ ]:
confusion_matrix_sklearn(bc_tuned_under,X_val_model,y_val)

In [ ]:
bc_tuned_val_under_perf = model_performance_classification_sklearn(
    bc_tuned_under,X_val_model,y_val
)
bc_tuned_val_under_perf

# Stacking

In [ ]:
estimators=[('Gradient',BaggingClassifier(random_state=1)),
           ('Randomforest',RandomForestClassifier(random_state=1))]
final_estimator=XGBClassifier(random_state=1)

In [ ]:
stk=StackingClassifier(estimators=estimators,final_estimator=final_estimator)

In [ ]:
stk.fit(X_train_model,y_train)

In [ ]:
confusion_matrix_sklearn(stk,X_train_model,y_train)

In [ ]:
stack_perf = model_performance_classification_sklearn(
    stk,X_train_model,y_train
)
stack_perf

In [ ]:
confusion_matrix_sklearn(stk,X_val_model,y_val)

# Tuning Gradient Boost

In [ ]:
%%time

gb_tuned = GradientBoostingClassifier(random_state=1)

parameters = {
    "init": [DecisionTreeClassifier(random_state=1,class_weight="balanced"),
            BaggingClassifier(random_state=1),
            AdaBoostClassifier(random_state=1,)],
    "n_estimators": np.arange(100,220,20),
    "subsample":[0.8,0.9,1],
    "max_features":[0.7,0.8,0.9,1],
}

# Type of scoring used to compare parameter combinations
scorer = metrics.make_scorer(metrics.accuracy_score)

#Calling RandomizedSearchCV
randomized_cv = RandomizedSearchCV(gb_tuned, parameters, scoring=scorer, cv=5, random_state=1, n_jobs = -1,verbose=2)

randomized_cv.fit(X_train_model, y_train)

gb_tuned=randomized_cv.best_estimator_

gb_tuned.fit(X_train_model, y_train)

In [ ]:
confusion_matrix_sklearn(gb_tuned,X_train_model,y_train)

In [ ]:
gb_tuned_train_perf = model_performance_classification_sklearn(
    gb_tuned,X_train_model,y_train
)
gb_tuned_train_perf

In [ ]:
confusion_matrix_sklearn(gb_tuned,X_val_model,y_val)

In [ ]:
gb_tuned_val_perf = model_performance_classification_sklearn(
    dt_tuned,X_val_model,y_val
)
gb_tuned_val_perf

In [ ]:
# KNN

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# Define random parameter distribution
param_dist = {
    'n_neighbors': np.arange(1, 21),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': np.arange(1, 3)
}

# Randomized Search
random_search = RandomizedSearchCV(estimator=knn, param_distributions=param_dist, n_iter=50, scoring='accuracy', cv=5, random_state=42, verbose=1)
random_search.fit(X_train_model, y_train)

# Best parameters and score
print("Best Parameters:", random_search.best_params_)
print("Best Accuracy Score:", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
